# Football Transfer Fee Prediction Pipeline

This notebook contains the data extraction, merging, feature engineering, and model training/evaluation for predicting football transfer fees. EDA and pairwise correlations have been removed for a cleaner production-ready pipeline.

## 1. Data Extraction

In [ ]:
import numpy as np
import pandas as pd
import os
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from fuzzywuzzy import process

# Load transfers data
transfers = pd.read_csv('../data/transfers.zip', compression='zip')

# Load FIFA ratings data for merging
ratings_zip_path = '../data/ratings.zip'
usecols = ['short_name', 'overall', 'potential', 'club']

fifa_frames = []
with zipfile.ZipFile(ratings_zip_path) as zf:
    for yr in [15, 16, 17, 18]:
        f_name = f'players_{yr}.csv'
        df_yr = pd.read_csv(zf.open(f_name), usecols=usecols)
        df_yr['Season'] = 2000 + yr
        fifa_frames.append(df_yr)

fifas_merged = pd.concat(fifa_frames, ignore_index=True)
fifas_merged['Lastname'] = fifas_merged['short_name'].str.split(' ').str[-1]
fifas_merged = fifas_merged.drop(columns=['short_name'])

## 2. Data Cleaning and Feature Engineering

In [ ]:
# Process Seasons
transfers['Season_transferred'] = transfers['Season'].str.split('-').str[0].astype('int64')
transfers = transfers.drop(columns=['Season'])

# Position cleaning
transfers.Position = transfers.Position.replace(
    to_replace=['Second Striker', 'Centre-Forward', 'Sweeper'],
    value=['Forward', 'Forward', 'Defender']
)

# Drop inconsistent rows (Age 0, specific positions to be dropped as per original analysis)
transfers_cleaned = transfers[~((transfers.Position == 'Midfielder') | (transfers.Position == 'Defender') | (transfers.Age == 0))].copy()

# Convert fees and market value to millions
transfers_cleaned['Transfer_fee_in_mln'] = transfers_cleaned['Transfer_fee'] / 1000000
transfers_cleaned['Market_value_in_mln'] = transfers_cleaned['Market_value'] / 1000000
transfers_cleaned = transfers_cleaned.drop(columns=['Transfer_fee', 'Market_value'])

# Prepare for merging: normalize names
def normalize_text(s):
    import unicodedata
    if not isinstance(s, str): return s
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('utf-8')

transfers_cleaned_2015_18 = transfers_cleaned[transfers_cleaned['Season_transferred'] > 2014].copy()
transfers_cleaned_2015_18['Lastname'] = transfers_cleaned_2015_18['Name'].apply(lambda x: x.split(' ')[-1])

for col in ['Lastname', 'Team_from', 'Team_to']:
    transfers_cleaned_2015_18[col] = transfers_cleaned_2015_18[col].apply(normalize_text)

fifas_merged['Lastname'] = fifas_merged['Lastname'].apply(normalize_text)
fifas_merged['club'] = fifas_merged['club'].apply(normalize_text)

# Club name mapping for consistency
club_map = {
    'Manchester United': 'Man Utd',
    'Manchester City': 'Man City',
    'Tottenham Hotspur': 'Spurs',
    'Olympique Lyonnais': 'Olympique Lyon',
    'Borussia Dortmund': 'Bor. Dortmund',
    'Wolverhampton Wanderers': 'Wolves',
    'FC Bayern München': 'Bayern Munich'
}
fifas_merged['club'] = fifas_merged['club'].replace(club_map)

## 3. Dataset Merging

In [ ]:
# Merge on Team_from first
transfers_with_fifa = transfers_cleaned_2015_18.merge(
    fifas_merged, 
    how='left', 
    left_on=['Lastname', 'Season_transferred', 'Team_from'], 
    right_on=['Lastname', 'Season', 'club']
)

# Fallback: Merge nulls on Team_to
transfers_null = transfers_with_fifa[transfers_with_fifa['club'].isnull()].copy()
transfers_null = transfers_null.drop(columns=['club', 'overall', 'potential', 'Season'])
transfers_null_merged = transfers_null.merge(
    fifas_merged, 
    how='left', 
    left_on=['Lastname', 'Season_transferred', 'Team_to'], 
    right_on=['Lastname', 'Season', 'club']
)

final_df = pd.concat([transfers_with_fifa.dropna(subset=['club']), transfers_null_merged.dropna(subset=['club'])], ignore_index=True)
final_df = final_df.drop(columns=['Name', 'Lastname', 'Season', 'club'])

## 4. Model Training and Evaluation

In [ ]:
# One-hot encoding for categorical variables
df_ml = pd.get_dummies(final_df)

y = df_ml['Transfer_fee_in_mln']
X = df_ml.drop(columns=['Transfer_fee_in_mln'])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=69, test_size=0.3)

lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred)}")
print(f"R2 Score: {r2_score(y_test, y_pred)}")